# SPFI ×4 — training on EMPS (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/main/notebooks/train_colab.ipynb)

Trains the SPFI model with the repo's `algo/` code **unchanged** (`algo.train.SPFITrainer`) on the
**train split only** of EMPS (`train.csv`). Test images are never extracted.

The result is a checkpoint file `spfi_x4_best.pt` in the same format the local app expects
(`model_state` + `hparams`), so you can use it locally for inference or for fine-tuning.

**Before you start:** Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")


## Settings

In [ ]:
from pathlib import Path

# ---- Code (the algo/ package is used exactly as it is in the repo) ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "main"

# ---- Data: public EMPS release (same images and train/test split as NanoVision's datasets/emps/EMPS) ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"

WORK_DIR = Path("/content")
CODE_DIR = WORK_DIR / "SR_amat_3"
DATA_DIR = WORK_DIR / "emps"
OUTPUT_DIR = WORK_DIR / "output"

# ---- Training (defaults are the ones in algo/train.py) ----
LR = 1e-4
WEIGHT_DECAY = 1e-2
PATIENCE = 20          # early stopping: validation checks (every 250 iterations) without improvement
SEED = 0

# Optional limits so the run ends before Colab disconnects. None = no limit (train until early stopping).
MAX_HOURS = 3.0
MAX_ITERS = None

# Copy the best checkpoint to Google Drive every time it improves, so a disconnect does not lose it.
SAVE_TO_DRIVE = True
DRIVE_DIR = Path("/content/drive/MyDrive/spfi_checkpoints")


## Code and dependencies

In [ ]:
# torch / torchvision / numpy / Pillow are preinstalled on Colab; add what algo/ needs on top.
!pip install -q "pytorch_wavelets>=1.3" "PyWavelets>=1.6" "setuptools<81"


In [ ]:
import hashlib
import subprocess
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def download_emps():
    """Download the pinned public EMPS archive once (no credentials needed) and check its SHA-256."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    return archive


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_split(split):
    """Extract train.csv, test.csv and only the images of `split` ('train' or 'test') into DATA_DIR/<split>/."""
    dest = DATA_DIR / split
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_emps()) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / f"{split}.csv").read_text())
        for image_id in ids:
            (dest / "images" / f"{image_id}.png").write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


import sys

if not CODE_DIR.exists():
    git("clone", "--depth", "1", "--branch", CODE_BRANCH, CODE_REPO_URL, str(CODE_DIR))
print("code:", git("log", "-1", "--format=%h %s", cwd=CODE_DIR).strip())

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))   # makes `algo` importable
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Dataset
EMPS is downloaded from its public release ([by256/emps](https://github.com/by256/emps), pinned commit),
so no GitHub token is needed. Its `train.csv` / `test.csv` are identical to the ones in NanoVision's
`datasets/emps/EMPS`.

Only the images listed in `train.csv` are extracted.

In [ ]:
EMPS_DIR, train_ids = extract_emps_split("train")
test_ids = read_split((EMPS_DIR / "test.csv").read_text())

on_disk = sorted(p.stem for p in (EMPS_DIR / "images").glob("*.png"))
assert on_disk == sorted(train_ids), "images/ must hold exactly the train split"
assert not set(on_disk) & set(test_ids), "test images must not be used for training"
print(f"{len(on_disk)} train images in {EMPS_DIR / 'images'} (test split: {len(test_ids)} images, not downloaded)")


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)


## Build the trainer
`SPFITrainer` splits the train images 80/20 into training / validation (for early stopping), so the
validation set also comes from the train split.

In [ ]:
from datetime import datetime

from algo.train import SPFITrainer

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
CKPT_DIR = WORK_DIR / "checkpoints" / stamp
RUN_DIR = WORK_DIR / "runs" / stamp       # metrics.jsonl / meta.json / samples/ (same format as the local UI)

trainer = SPFITrainer(
    emps_dir=EMPS_DIR,
    ckpt_dir=CKPT_DIR,
    lr=LR,
    wd=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    run_dir=RUN_DIR,
)
print("parameters:", sum(p.numel() for p in trainer.model.parameters()))


## Progress line every 250 iterations
Replaces the trainer's own validation print with one line that also shows SSIM:
the train loss / SSIM are averaged over the last 250 iterations, the `val_*` values are on the validation set.
`*` marks a new best checkpoint.

In [ ]:
import algo.train as train_module
from algo.run_logger import RunLogger

_window = {"n": 0, "loss": 0.0, "ssim_loss": 0.0}


def _log_train(step, loss, charbonnier, ssim_loss, lr):
    # Sum detached tensors so the GPU only syncs once per validation
    _window["n"] += 1
    _window["loss"] = _window["loss"] + loss.detach()
    _window["ssim_loss"] = _window["ssim_loss"] + ssim_loss.detach()
    RunLogger.log_train(trainer.logger, step, loss, charbonnier, ssim_loss, lr)


def _log_val(step, val_loss, val_psnr, val_ssim, is_best, no_improve):
    n = max(_window["n"], 1)
    loss, ssim = float(_window["loss"]) / n, 1.0 - float(_window["ssim_loss"]) / n
    _window.update(n=0, loss=0.0, ssim_loss=0.0)
    print(
        f"[iter {step:7d}]  loss={loss:.4f}  ssim={ssim:.4f}"
        f"  |  val_loss={val_loss:.4f}  val_psnr={val_psnr:.2f} dB  val_ssim={val_ssim:.4f}"
        f"  |  best_val_loss={trainer.best_val_loss:.4f}  no_improve={no_improve}/{PATIENCE}"
        f"{'  *' if is_best else ''}"
    )
    RunLogger.log_val(trainer.logger, step, val_loss, val_psnr, val_ssim, is_best, no_improve)


def _print(*args, **kwargs):
    # Drop the trainer's own "[iter ...]" line; _log_val prints the full one
    if not (args and str(args[0]).startswith("[iter ")):
        print(*args, **kwargs)


# Instance attributes over the class methods, so re-running this cell does not stack wrappers
trainer.logger.log_train = _log_train
trainer.logger.log_val = _log_val
train_module.print = _print

## Train
Runs until early stopping, `MAX_HOURS` or `MAX_ITERS`, whichever comes first. A checkpoint is saved and
the model is validated every 250 iterations.

You can also press **Stop** (■) at any time: the trainer stops cleanly and you can go on to *Export*.
Running this cell again continues from where it stopped.

In [ ]:
import shutil
import threading
import time

stop_event = threading.Event()
done = threading.Event()
best_ckpt = CKPT_DIR / "spfi_x4_best.pt"


def watch(started=time.time()):
    """Stop the trainer at the time / iteration limit and mirror the best checkpoint to Drive."""
    mirrored = None
    while not done.wait(5):
        if MAX_ITERS is not None and trainer.iteration >= MAX_ITERS:
            stop_event.set()
        if MAX_HOURS is not None and time.time() - started >= MAX_HOURS * 3600:
            stop_event.set()
        if SAVE_TO_DRIVE and best_ckpt.exists():
            mtime = best_ckpt.stat().st_mtime
            if mtime != mirrored:
                time.sleep(1)   # let torch.save finish writing
                shutil.copy2(best_ckpt, DRIVE_DIR / f"spfi_x4_best_{stamp}.pt")
                mirrored = mtime


watcher = threading.Thread(target=watch, daemon=True)
watcher.start()
try:
    trainer.train(stop_event=stop_event)
finally:
    done.set()
    watcher.join()
print(f"finished at iteration {trainer.iteration}, best val loss {trainer.best_val_loss:.4f}")


## Export the weights
Saves the current weights as `spfi_x4_final.pt` and picks `spfi_x4_best.pt` (lowest validation loss)
when it exists. Then checks that the file loads the way the local app loads it, and downloads it.

In [ ]:
trainer._save_checkpoint("final")   # same format as the periodic checkpoints
final_ckpt = CKPT_DIR / "spfi_x4_final.pt"
chosen = best_ckpt if best_ckpt.exists() else final_ckpt

WEIGHTS_FILE = OUTPUT_DIR / "spfi_x4_best.pt"
shutil.copy2(chosen, WEIGHTS_FILE)

# Same checks as the local app's inference loader (inference/runner.py::_load_model)
from algo.model import build_spfi

ckpt = torch.load(WEIGHTS_FILE, map_location="cpu", weights_only=True)
hp = ckpt["hparams"]
build_spfi(scale=hp["scale"], embed_dim=hp["embed_dim"]).load_state_dict(ckpt["model_state"])
print(f"{WEIGHTS_FILE.name}: from {chosen.name}, iteration {ckpt['iteration']}, "
      f"val loss {ckpt['best_val_loss']:.4f}, hparams {hp}")

logs_zip = shutil.make_archive(str(OUTPUT_DIR / f"spfi_run_{stamp}"), "zip", RUN_DIR)
if SAVE_TO_DRIVE:
    shutil.copy2(WEIGHTS_FILE, DRIVE_DIR / f"spfi_x4_best_{stamp}.pt")
    shutil.copy2(logs_zip, DRIVE_DIR)
    print("copied to", DRIVE_DIR)


In [ ]:
from google.colab import files

files.download(str(WEIGHTS_FILE))   # the weights file to use locally
files.download(logs_zip)            # optional: training curves / samples (unzip into the local runs/ folder)


## Using the weights locally
* **Inference tab** of the local app: set the model path to the downloaded `spfi_x4_best.pt`.
* **Fine-tuning:** `python -m algo.train --emps_dir <EMPS> --init finetune --weights spfi_x4_best.pt`.
* **Test PSNR / SSIM / FLOPs:** `inference_colab.ipynb`.